# Progetto 8 — Self-Supervised Latent Representations for Imbalanced Apical Periodontitis GradingComputer Vision A.A. 2025-2026 · Prof. Irene Amerini · Sapienza / ALCOR Lab**Gruppo:** Nome 1 (matricola) · Nome 2 (matricola) · Nome 3 (matricola)---Il notebook segue la struttura concettuale richiesta dal corso(*Imports → Globals → Utils → Data → Network → Train → Evaluation*).Le celle importano dai moduli della repo invece di duplicarne il codice: ognisezione corrisponde al file omonimo, e la logica resta in un posto solo.**Prima di lanciare qualsiasi training, eseguite la sezione Data e guardate lestatistiche bbox.** Se la lesione mediana copre meno di ~4 token, larisoluzione scelta rende il downstream privo di segnale.

## Setup — repo e GPUAggiornate `REPO_URL` con la vostra repo.

In [ ]:
REPO_URL = "https://github.com/UTENTE/cv-periapical-jepa.git"import os, subprocess, sysif not os.path.isdir("/kaggle/working/repo"):    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/repo"], check=True)else:    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], check=False)sys.path.insert(0, "/kaggle/working/repo")os.chdir("/kaggle/working/repo")print("cwd:", os.getcwd())

In [ ]:
import torchprint("PyTorch :", torch.__version__)print("CUDA    :", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print(f"GPU     : {p.name}  {p.total_memory/1024**3:.1f} GB")    # T4 (Turing) supporta bene fp16; P100 (Pascal) no -> AMP rende poco.    print("AMP fp16 conveniente:", p.major >= 7)

## Imports

In [ ]:
import numpy as npimport torchimport matplotlib.pyplot as pltimport globals as Gimport utils, data, network, imbalance, evaluationfrom train_ssl import train as train_sslfrom train_downstream import cache_latents, load_latents, train_head, run_grid

## GlobalsTutti gli iperparametri stanno in `globals.py`. Modificateli lì, non qui: è ciò che tiene gli esperimenti riproducibili.

In [ ]:
print("Device          :", G.DEVICE)print("Su Kaggle       :", G.ON_KAGGLE)print("Data root       :", G.DATA_ROOT)print("Tile            :", G.TILE_SIZE, "stride", G.TILE_STRIDE)print("Patch           :", G.PATCH_SIZE)print("Backbone        :", G.DEFAULT_VARIANT, G.VIT_VARIANTS[G.DEFAULT_VARIANT])print("Epoche SSL      :", G.SSL_EPOCHS, "batch", G.SSL_BATCH_SIZE)print("Classi PAI      :", G.PAI_GRADES, "attese", G.EXPECTED_COUNTS)

## UtilsVerifica del monitoraggio del collasso su casi con esito noto.È il meccanismo che vi avvisa se il pre-training sta fallendo silenziosamente.

In [ ]:
utils.set_seed()D, N = 192, 512casi = {    "isotropo (sano)":     torch.randn(N, D),    "costante (collasso)": torch.randn(1, D).repeat(N, 1) + 1e-6*torch.randn(N, D),    "rango 8":             torch.randn(N, 8) @ torch.randn(8, D),}for nome, e in casi.items():    print(f"{nome:22s} std={utils.embedding_std(e):.6f}  eff_rank={utils.effective_rank(e):7.2f} / {D}")

## DataTre passaggi, in quest'ordine. Il secondo decide l'architettura.

In [ ]:
data.inspect_dataset()

In [ ]:
records = data.parse_annotations(verbose=True)

### Statistiche bbox — la verifica che decide tuttoUna panoramica inquadra l'intera arcata; una lesione è di pochi millimetri. Se ridimensionate l'immagine intera a 224×224 la lesione finisce sotto la dimensione di un patch token e il latente estratto alla bbox non contiene la patologia.

In [ ]:
stats = data.bbox_statistics(records)

In [ ]:
splits = data.build_splits(records)

### Un'occhiata ai datiGuardate sempre qualche campione prima di addestrare.

In [ ]:
ds = data.LesionCropDataset(records, splits["train"])fig, axes = plt.subplots(2, 5, figsize=(15, 6))for ax, i in zip(axes.flat, np.random.choice(len(ds), 10, replace=False)):    s = ds[i]    img = s["image"][0].numpy() * 0.5 + 0.5    ax.imshow(img, cmap="gray")    x0, y0, x1, y1 = s["bbox"].tolist()    ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color="#00ff00", lw=1.5))    ax.set_title(f"PAI {G.PAI_GRADES[s['label']]}", fontsize=10)    ax.axis("off")fig.suptitle("Crop di lesione con bbox", y=1.0)fig.tight_layout(); plt.show()

## NetworkI-JEPA: context encoder, target encoder aggiornato via EMA, predictor shallow — la struttura richiesta dall'obiettivo 1 del brief.

In [ ]:
model = network.build_ijepa(G.DEFAULT_VARIANT)x = torch.randn(2, 3, G.TILE_SIZE, G.TILE_SIZE)loss, emb = model(x)print(f"{G.DEFAULT_VARIANT}: {network.count_params(model)/1e6:.2f}M parametri")print(f"griglia {model.grid}x{model.grid}  loss={loss.item():.4f}  emb={tuple(emb.shape)}")bbox = torch.tensor([[40., 40., 120., 120.]])print("token dentro la bbox:", network.bbox_to_token_mask(bbox, model.grid).sum().item())

## Train — Stadio 1: pre-training self-supervised**Guardate il monitoraggio, non la loss.** La loss I-JEPA può scendere regolarmente mentre gli embedding collassano a una costante: predire un target costante è banale.Su Kaggle la sessione si stacca a 12 h e i checkpoint sono salvati a ogni epoca — se vi disconnette, rilanciate con `resume=True`.

In [ ]:
# smoke test: 20 step, verifica che tutto giri prima di impegnare ore di GPU_ = train_ssl(G.DEFAULT_VARIANT, epochs=1, smoke=True)

In [ ]:
model = train_ssl(G.DEFAULT_VARIANT, epochs=G.SSL_EPOCHS, resume=True)

## Train — Stadio 2: caching dei latenti e classificazioneI latenti si estraggono **una volta sola**. Da lì ogni esperimento sullo sbilanciamento gira in secondi, anche su CPU: l'ablation diventa gratuito.

In [ ]:
cache_latents(G.DEFAULT_VARIANT, arm="ijepa")# Il braccio critico: senza questo confronto non avete dimostrato che il# pre-training in-domain serva a qualcosa. Vedi ANALISI_PROGETTO_8.md §9.# cache_latents(G.DEFAULT_VARIANT, arm="imagenet")

In [ ]:
cached = load_latents(G.DEFAULT_VARIANT, "ijepa")clf, best = train_head(cached, method="class_weighted", head_type="ordinal", verbose=True)

## EvaluationIl brief vieta implicitamente l'accuracy globale: con il 61% di PAI 3, predire sempre la maggioritaria dà 61% e zero utilità clinica.Oltre alle metriche richieste c'è il **kappa quadratico pesato**, perché il PAI è una scala ordinale e confondere PAI 3 con PAI 5 è clinicamente peggio che confondere 4 con 5.

In [ ]:
res = evaluation.evaluate_split(clf, cached["data"]["test"], "ordinal")evaluation.print_report(res, "JEPA in-domain / class_weighted / ordinale")

In [ ]:
print("figura:", evaluation.plot_confusion(res, "confusion_test"))

### Ablation completoMetodi × teste × seed. Con sbilanciamento 7:1 i margini sono stretti: servono più seed e intervalli di confidenza, non un run singolo.

In [ ]:
rows = run_grid(G.DEFAULT_VARIANT, arm="ijepa")